# Lab 04: Graphical Models and D-Separation

_Note: Questions marked with (*) can be presented in the exercise sessions for up to 1 bonus point._

<hr>

We saw in the lecture that the conditional independence properties of a distribution that factorizes over a directed acyclic graph (DAG) can be deduced from properties of the graph. We introduced the concept of D-separation as a necessary and sufficient condition on the graph to have independence of a set of variables $X_A$ to another set $X_B$ conditionally to $X_C$. 

The purpose of this lab is to implement two methods to verify this conditional independence.

## Detection of Cardiovascular Diseases

We consider the following graphical model describing relations between different cardiovascular diseases.

![alt text](cardiovascular.jpg)

(Image source: https://bmjopen.bmj.com/content/10/5/e035867)

#### Question 1 (*)
a) Show that "Hypertension" is independent of "Has a chronic condition". <br>
b) Are the variables "Current smoker" and "Has a chronic condition" dependent or independent? Does it change when conditioned on "History of high cholesterol"?

Note that in the previous questions, it was not necessary to know the distribution, just that it factorizes over the graph.

## Implementation of an Algorithm for D-separation

In [2]:
from dag import DAGNode, DAG, parse_graph

### Directed Acyclic Graphs

Consider the `DAGNode` and `DAG`. A `DAGNode` has the following fields:
```python
name: str
children: set[DAGNode]
parents: set[DAGNode]
```
Meanwhile, a `DAG` consists of the following fields:
```python
V: dict[str, DAGNode]
roots: list[str]
```

The function `parse_graph` can be used to quickly create a DAG by providing a list of edges.

Usage notes:
* Statements are separated by `;`.
* Each statement defines a sequence of edges.
* `(n1, n2) -> (n3, n4, n5)` create edges from both `n1` and `n2` to all of `n3`, `n4` and `n5`, so 6 edges in total. This can be extended to arbitrarily many nodes.
* `n1` is short for `(n1)`.
* You can use arbitrary names for nodes, so long as they do not contain any of `;`, `->` or `,`.

Example usage:
```python
g = parse_graph('X1 -> (X2, X5); X2 -> X3 -> X4 -> X7; X5 -> X6 -> X7')
```
This creates the below DAG (also on slide 18 of Lecture 4).

<div>
<img src="graph.png" width="500"/>
</div>

#### Question 2
Consider the DAG corresponding to $A \leftarrow B \rightarrow C \rightarrow D$.<br>
a) Implement it using the `parse_graph` method.<br>
b) Do we have that $A$ is independent of $D$ given $B$?

In [ ]:
g = parse_graph('B -> (A, C); C -> D')

### Blocked chains
Let $G = (V,E)$ be a DAG. Denote the descendants of a node $v \in V$ by $\mathsf{desc}(v)$.
Recall the following notions from the lecture:

A *chain* from $a$ to $b$ is a sequence of nodes $(v_0,v_1,\dots,v_n)$ with $v_0=a$ and $v_n=b$ that form a simple (i.e. without repeating nodes) path in the undirected graph induced by $G$.

A chain is said to be *blocked* by $C \subset V$, if either of the following two conditions is fulfilled:

1. there is a V-structure, $v_{i-1} \rightarrow v_i \leftarrow v_{i+1}$ and neither $v_i$ nor any of its descendants are in $C$: $$(\{v_i\} \cup \mathsf{desc}(v_i)) \cap C = \varnothing,$$
2. one of the nodes in the chain that is not part of a V-structure is contained in $C$: there is an $i \in \{1,\dots,n-1\}$ such that $v_i \in C$ and we do not have $v_{i-1} \rightarrow v_i \leftarrow v_{i+1}$.

In general, for mutually disjoint sets $A,B,C \subset V$, we say that $A$ and $B$ are d-separated by $C$, if **all chains** between members of $A$ and $B$ are blocked by $C$.

#### Question 3 (*)
a) Extend the implementation of `DAGNode` in the file `dag.py` by a function for computing its descendants. The return type should be `set[DAGNode]`.<br>
b) Implement the function `is_blocked_chain(g: DAG, chain: list[str], C: set[str]) -> bool` which should return whether the given chain is blocked by $C$.

In [4]:
def is_blocked_chain(chain: list[DAGNode], C: set[DAGNode]) -> bool:
    for l, m, r in zip(chain, chain[1:], chain[2:]):
        if m in l.children and m in r.children:
            return (m not in C and m.descendants().isdisjoint(C))
        elif m in C:
            return True
    return False

Below is a function that generates all chains between two nodes, as well as a function that checks D-separation by using your implementation of `is_blocked_chain`. You do not have to modify these functions, but you can investigate how they operate, if you like.

Since we need to check whether all chains are blocked, we need a way of generating all chains. This is done below in the method `all_chains`. You do not have to modify it.

In [11]:
def all_chains(a: DAGNode, b: DAGNode) -> list[list[DAGNode]]:
    """
    Returns a list of all chains from a to b in a DAG.
    """
    def dfs(v: DAGNode, hist: list[DAGNode] = []):
        ret = []
        nbs = v.children.union(v.parents)
        if not nbs:
            return []
        for c in nbs:
            if c in hist: # Cycle detected
                continue
            if c == b:
                return [hist + [v,b]]
            chain = dfs(c, hist + [v])
            if chain:
                ret += chain
        return ret
    return dfs(a)

#### Question 4 (*)
a) Implement the `d_sep` method below, which should check whether the node sets $A$ and $B$ are conditionally independent, given $C$.<br>
b) Verify your answer to Question 2.b) using `d_sep`. Further, implement the DAG for cardiovascular disease and verify your answers to Questions 1 and 2.

In [16]:
def d_sep(g: DAG, A: set[str], B: set[str], C: set[str]) -> bool:
    for a in A:
        for b in B:
            for chain in all_chains(g.V[a], g.V[b]):
                for c in C:
                    if not is_blocked_chain(chain, g.V[c]):
                        return False
    return True

In [17]:
d_sep(g, set('A'), set('D'), set('B'))

TypeError: argument of type 'DAGNode' is not iterable

#### Question 5 (*)
Show that the worst-case runtime of `d_sep` is exponential in the size of the graph. _(Hint: One of the helper functions is the culprit.)_

### The Bayes-Ball Algorithm
Previously, we have implemented methods for testing d-separation for specified node groups $A$, $B$ and $C$. What if we only have $A$ and $C$, and want to find the largest set of nodes $B$ which is d-separated from $A$, by $C$? For this, we can use the _Bayes-ball algorithm_. Despite completing a bigger task than `d_sep`, its runtime is _linear_ in the size of the graph.

The core idea is that a ball is thrown from all nodes in $A$ to its children and parents. Whenever a regular node receives a ball from a parent, it will pass it to all of its its children. But if that node was in $C$, it will instead deflect the ball back to its parents. When a regular node receives a ball one of its child instead, it will send it to both its children and parents. But if that node was in $C$, it will instead do nothing. After ensuring that no throw is performed more than once, the algorithm will terminate.

The algorithm proceeds as follows (adapted from [1]):
>1. Initialize all nodes as neither visited, nor marked on the top, nor marked on the bottom.
>2. Create a schedule of nodes to be visited, initialized with each node in $A$ to be visited as if from one of its children.
>3. While there are still nodes scheduled to be visited:
>    1. Pick any node $v$ scheduled to be visited and remove it from the schedule. Either $v$ was scheduled for a visit from a parent, a visit from a child, or both.
>    2. Mark $v$ as visited.
>    3. If $v \not\in C$ and the visit to $v$ is from a child:
>        1. if the top of $v$ is not marked, then mark its top and schedule each of its parents to be visited;
>        2. if the bottom of $v$ is not marked, then mark its bottom and schedule each of its children (if any) to be visited.
>    4. If the visit to $v$ is from a parent:
>        1. If $v \in C$ and the top of $v$ is not marked, then mark its top and schedule each of its parents to be visited;
>        2. if $v \not\in C$ and the bottom of $v$ is not marked, then mark its bottom and schedule each of its children to be visited.

Once the algorithm finishes, the nodes which are not marked on the bottom (i.e. have never sent the ball to their children), constitutes the set $B$ of nodes that are d-separated from $A$, by $C$.

#### Question 6 (*)
(a) Implement the Bayes-ball algorithm. You can use a queue to create the schedule, by importing `from collections import deque`. A `deque` is a double-ended queue, for which it is cheap to `append` and `pop` from the right and the left (with `appendleft` and `popleft`).<br>
(b) Test the algorithm on the cardiovascular disease graph.

In [ ]:
def bayes_ball(g: DAG, A: set[str], C: set[str]):
    pass

## References
[1] Ross D. Shachter. _Bayes-Ball: The Rational Pastime (for Determining Irrelevance and Requisite Information in Belief Networks and Influence Diagrams)_ (arXiv: https://arxiv.org/pdf/1301.7412)